In [ ]:
"""
MahaRERA Proper CSV and Excel Merger

Key Features:
1. Aligns all input CSV files against the unified schema (all 54 columns + source traceability).
2. Preserves split columns (bhk_details, bhk_details_part2..5, All_Parking_Details_part1..3, etc.)
   in their dedicated columns, ensuring cells never exceed Excel's 32,767 character limit.
3. Does NOT mangle cell values into JSON fragments like [{"part": ...}].
4. Handles large columns safely: ensures each part is kept intact and clean.
5. Produces:
   - maharera_merged.csv (UTF-8 with BOM for Excel compatibility)
   - maharera_merged.xlsx (Native Excel format, 100% complete with 0 truncation)
   - merge_report.csv (Row counts and file status)
"""

from pathlib import Path
import csv
import pandas as pd
import openpyxl

INPUT_FOLDER = Path(r"D:\Mumbai\Thane Output files")
OUTPUT_FOLDER = Path(r"D:\Mumbai\Thane Output files")

OUTPUT_CSV = OUTPUT_FOLDER / "maharera_merged.csv"
OUTPUT_XLSX = OUTPUT_FOLDER / "maharera_merged.xlsx"
OUTPUT_REPORT = OUTPUT_FOLDER / "merge_report.csv"

# Global target schema with logical column order
TARGET_COLUMNS = [
    'Registration Number', 'Date of Registration', 'Rera ID', 'Project Location',
    'Project Type', 'Project Name', 'survey_no', 'Permissible Builtup Area',
    'Sanctioned Builtup Area', 'Total Land Area', 'Aggregate area',
    'Land Area for Project Applied for this Registration',
    'Project proposed completion date original', 'Project proposed completion date revised',
    'Longitude', 'Latitude', 'Are there any Investor other than the Promoter in the Project',
    'State_UT', 'District', 'Taluka', 'Village', 'Pin_Code',
    'Boundaries_East', 'Boundaries_West', 'Boundaries_South',
    'Boundaries_North', 'Boundaries_North_part2',
    'Promoter_Details', 'Promoter_Details_part2', 'Promoter_Details_part3',
    'Promoter_Details_part4', 'Promoter_Details_part5',
    'bhk_details', 'bhk_details_part2', 'bhk_details_part3',
    'bhk_details_part4', 'bhk_details_part5',
    'building_details', 'Summary_of_Apartments/Units', 'past_experience_info',
    'Litigations related to the project ?',
    'All_Parking_Details', 'All_Parking_Details_part2', 'All_Parking_Details_part3',
    'Bank Name', 'IFSC Code', 'Bank Address', 'ProjectStatus', 'Locality',
    'Organization', 'Last_Modified', 'Commencing From', 'Leftmost Date', 'Ending With'
]


def main():
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    
    # Exclude output files from reading
    excluded = {OUTPUT_CSV.resolve(), OUTPUT_XLSX.resolve(), OUTPUT_REPORT.resolve()}
    input_files = sorted(
        p for p in INPUT_FOLDER.glob("Thane_scrape_output_*.csv")
        if p.is_file() and p.resolve() not in excluded
    )
    
    if not input_files:
        raise FileNotFoundError(f"No input CSV files found matching Thane_scrape_output_*.csv in {INPUT_FOLDER}")

    print(f"Found {len(input_files)} input CSV file(s).")
    
    # Verify any new or extra columns across input files
    all_seen_columns = list(TARGET_COLUMNS)
    for p in input_files:
        df_head = pd.read_csv(p, nrows=0, encoding="utf-8-sig", dtype=str)
        for col in df_head.columns:
            cleaned = col.strip().replace("\ufeff", "")
            if cleaned not in all_seen_columns:
                all_seen_columns.append(cleaned)
                
    final_columns = all_seen_columns + ["source_file", "source_row"]
    print(f"Total unified columns: {len(final_columns)}")
    
    report = []
    total_rows = 0
    all_frames = []
    
    for path in input_files:
        print(f"\nProcessing: {path.name}")
        try:
            df = pd.read_csv(path, dtype=str, encoding="utf-8-sig", keep_default_na=False)
            file_rows = len(df)
            
            # Clean column names
            df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]
            
            # Add source tracking
            df["source_file"] = path.name
            df["source_row"] = range(2, file_rows + 2)
            
            # Reindex to final unified schema, filling missing columns with empty string
            df_reindexed = df.reindex(columns=final_columns, fill_value="")
            
            # Check cell lengths for any cell exceeding 32,700 chars
            for col in final_columns:
                lengths = df_reindexed[col].astype(str).str.len()
                max_l = lengths.max()
                if max_l > 32700:
                    print(f"  [ALERT] Column {col} in {path.name} has cells of length {max_l}")

            all_frames.append(df_reindexed)
            total_rows += file_rows
            
            report.append({
                "file_name": path.name,
                "rows_merged": file_rows,
                "status": "SUCCESS",
                "error": ""
            })
            print(f"  Successfully processed {file_rows:,} rows")
            
        except Exception as exc:
            report.append({
                "file_name": path.name,
                "rows_merged": 0,
                "status": "ERROR",
                "error": str(exc)
            })
            print(f"  [ERROR] {path.name}: {exc}")

    # Combine all dataframes
    merged_df = pd.concat(all_frames, ignore_index=True)
    print(f"\nTotal merged records: {len(merged_df):,} rows x {len(merged_df.columns)} columns")
    
    # 1. Write Merged CSV
    print(f"Writing merged CSV to {OUTPUT_CSV}...")
    merged_df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_MINIMAL
    )
    print("CSV write complete.")
    
    # 2. Write Merged Excel
    print(f"Writing merged Excel to {OUTPUT_XLSX}...")
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        merged_df.to_excel(
            writer,
            sheet_name="MahaRERA_Merged",
            index=False
        )
    print("Excel write complete.")
    
    # 3. Write Merge Report
    pd.DataFrame(report).to_csv(OUTPUT_REPORT, index=False, encoding="utf-8-sig")
    print(f"Merge report saved to {OUTPUT_REPORT}.")
    
    print("\n================ MERGE VERIFICATION ================")
    print(f"Total Rows:    {len(merged_df):,}")
    print(f"Total Columns: {len(merged_df.columns):,}")
    print("Large columns preserved properly:")
    for base in ["bhk_details", "All_Parking_Details", "Promoter_Details", "Boundaries_North"]:
        parts = [c for c in merged_df.columns if c == base or c.startswith(base + "_part")]
        non_empty = {p: (merged_df[p] != "").sum() for p in parts}
        print(f"  {base}: {parts}")
        print(f"    Counts with data: {non_empty}")
    print("=====================================================")


if __name__ == "__main__":
    main()
